In [1]:
from pathlib import Path
from datetime import datetime
from sklearn.preprocessing import QuantileTransformer
from matplotlib.backends.backend_pdf import PdfPages

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [2]:
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

In [3]:

# ------------------------- Basisverzeichnis (aktuelles Notebook-Verzeichnis) -------------------------
base_dir = Path.cwd()

# ------------------------- Pfad zur "Kompletten Datenbank" setzen -------------------------
db_root = (
    base_dir.parent.parent 
    / "1_Acquisition" 
    / "1.1_Data-Acquisition-Wrapper"
    / "Angepasste_Datenbanken"
    / "Nach_IBF-und-Temp_Filter"
    / "Komplette_Datenbank"
)

# ------------------------- Alle Unterordner lesen, Ordner mit "neustem" Datum setzen -------------------------
if not db_root.exists():
     raise FileNotFoundError(f"Datenbank-Pfad nicht gefunden: {db_root}")

timestamp_folders = [f for f in db_root.iterdir() if f.is_dir()]
if not timestamp_folders:
    raise FileNotFoundError(f"Keine Datenbank-Ordner in {db_root} gefunden.")

latest_folder = max(timestamp_folders, key=lambda f: f.stat().st_mtime)

# ------------------------- Komplette Datenbank als CSV -------------------------
input_path = latest_folder / "Komplette_Datenbank.csv"

print("Verwendeter Datenbankpfad:")
print(input_path)

Verwendeter Datenbankpfad:
F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\Nach_IBF-und-Temp_Filter\Komplette_Datenbank\2026-02-03_15-26-22\Komplette_Datenbank.csv


In [4]:
# ------------------------- Ordner "Preprocessing" zum Abspeichern -------------------------
output_root = base_dir / "Preprocessing"
output_root.mkdir(exist_ok=True) 

# ------------------------- Zeitstempel erzeugen -------------------------
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_dir = output_root / timestamp
output_dir.mkdir(parents=True, exist_ok=False)

print("Erzeugter Output-Ordner:")
print(output_dir)

Erzeugter Output-Ordner:
F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\3_Machine-Learning\3.1_Preprocessing\Preprocessing\2026-02-03_15-26-41


<div class="alert alert-info">
    <b>Preprocessing & Scaling</b></b><br><br>
    <b>1. Log-Transformation</b><br>
    - Anwendung von np.log1p(x) (ln(1 + x)).<br>
    - Verhindert negative Werte für kleine Zahlen (x < 1)<br>
    <br>
    <b>2. Robust Scaling (IQR)</b><br>
    - Zentriert auf Median, skaliert mit IQR (Ausreißer-resistent).<br>
    - Macht Daten statistisch ideal für die SOM<br>
</div>

In [5]:
# ------------------------- Daten laden -------------------------
df = pd.read_csv(input_path, low_memory=False)

print(f"Datensatz geladen mit {df.shape[0]} Zeilen und {df.shape[1]} Spalten.")

Datensatz geladen mit 94264 Zeilen und 31 Spalten.


In [6]:
# ------------------------- Attribute definieren -------------------------

ALL_FEATURES_SCHEMA = [
    # ----------------- Hauptionen -----------------
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L", 
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
    
    # ----------------- Physikalische Parameter -----------------
    "temperature_in_c",
    "pH",
    "electrical_conductivity_25c_in_uS/cm",
    "redox_potential_in_mV",
    "total_dissolved_solids_in_mmol/L",

    # ----------------- Weitere Ionen / Spurenelemente --------------------
    "K_in_mmol/L",
    "NO3_in_mmol/L",
    "Li_in_mmol/L", 
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "HS_in_mmol/L",
    "O2_in_mmol/L",
    "Sr_in_umol/L",
    "NH4_in_umol/L",
    "F_in_umol/L",
    "H2SiO3_in_umol/L",
    
    # ----------------- Metadaten -----------------
    "depth_bgl_in_m",
    "rock_type",
    "stratigraphic_period"
]

print(f"Definiertes Global-Schema: {len(ALL_FEATURES_SCHEMA)} Attribute")


# ------------------------- Mapping & Vorbereitung -------------------------

features_to_process = []

for schema_name in ALL_FEATURES_SCHEMA:
    found_col = None
    # ------------------------- Exakt -------------------------
    if schema_name in df.columns:
        found_col = schema_name
    else:
        # ------------------------- Basis-Suche (falls Einheit abweicht) -------------------------
        if '_in_' in schema_name:
            base = schema_name.split('_in_')[0]
            candidates = [c for c in df.columns if c.startswith(base + '_in_') or c == base]
            if candidates:
                found_col = candidates[0]
    
    if found_col:
        features_to_process.append(found_col)

features_to_process = list(set(features_to_process))

# ------------------------- Metadata Spalten dürfen nicht in features_to_process sein -------------------------
metadata_cols_to_exclude = ["rock_type", "stratigraphic_period"]
for m_col in metadata_cols_to_exclude:
    if m_col in features_to_process:
        features_to_process.remove(m_col)
        print(f"'{m_col}' aus numerischer Verarbeitung entfernt.")

print(f"Gefundene Spalten im Dataset: {len(features_to_process)} / {len(ALL_FEATURES_SCHEMA)}")
print(features_to_process)


# ------------------------- Filterung von Metadata -------------------------
# ------------------------- 'Unknown' bei Stratigraphie entfernen (zu NaN setzen) -------------------------
if 'stratigraphic_period' in df.columns:
    before = df['stratigraphic_period'].notna().sum()
    df.loc[df['stratigraphic_period'] == 'Unknown', 'stratigraphic_period'] = np.nan
    after = df['stratigraphic_period'].notna().sum()
    print(f"Filterung Stratigraphie: {before - after} 'Unknown' Werte entfernt.")


# ------------------------- Bereinigung nicht-numerischer Werte -------------------------
print("Bereinige Daten (Coercing to Numeric)...")
for col in features_to_process:
    # Erzwinge numerische Typen, nicht-konvertierbare Werte (z.B. '—') werden NaN
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("Bereinigung abgeschlossen.")


# ------------------------- df_clean ist df -------------------------
df_clean = df.copy()

# ------------------------- Update Kanidatenliste für Skalierung -------------------------
transform_candidates = [c for c in features_to_process if pd.api.types.is_numeric_dtype(df[c])]
print(f"Bereit für Transformation: {len(transform_candidates)} Spalten.")

Definiertes Global-Schema: 25 Attribute
'rock_type' aus numerischer Verarbeitung entfernt.
'stratigraphic_period' aus numerischer Verarbeitung entfernt.
Gefundene Spalten im Dataset: 23 / 25
['Fe_in_mmol/L', 'Li_in_mmol/L', 'Mg_in_mmol/L', 'NO3_in_mmol/L', 'Mn_in_mmol/L', 'Cl_in_mmol/L', 'H2SiO3_in_umol/L', 'Ca_in_mmol/L', 'Na_in_mmol/L', 'total_dissolved_solids_in_mmol/L', 'K_in_mmol/L', 'depth_bgl_in_m', 'F_in_umol/L', 'temperature_in_c', 'NH4_in_umol/L', 'redox_potential_in_mV', 'HCO3_in_mmol/L', 'O2_in_mmol/L', 'SO4_in_mmol/L', 'HS_in_mmol/L', 'electrical_conductivity_25c_in_uS/cm', 'pH', 'Sr_in_umol/L']
Filterung Stratigraphie: 60479 'Unknown' Werte entfernt.
Bereinige Daten (Coercing to Numeric)...
Bereinigung abgeschlossen.
Bereit für Transformation: 23 Spalten.


In [7]:
# ------------------------- Log-Transformation Report -------------------------

skewness = df[transform_candidates].skew().sort_values(ascending=False)
skewed_cols = skewness[abs(skewness) > 1.0].index.tolist()

df_base = df.copy()
base_cols = []

pdf_log_path = output_dir / "Log_Transformation_Report.pdf"

with PdfPages(pdf_log_path) as pdf:
    fig = plt.figure(figsize=(10, 6))
    plt.text(0.5, 0.5, f"Log-Transformation Report\n(Applied independent of full completeness)", 
             ha='center', va='center', fontsize=16)
    plt.axis('off')
    pdf.savefig(fig)
    plt.close()

    for col in transform_candidates:
        # ------------------------- Spaltenweise, NaNs in anderen Spalten egal -------------------------
        series = df[col].dropna()
        if len(series) == 0: continue
        
        # ------------------------- Schiefheit überprüfen -------------------------
        if col in skewed_cols and series.min() >= 0:
            new_col = f"{col}_log"
            df_base[new_col] = np.log1p(df_base[col])
            base_cols.append(new_col)
            
            # ------------------------- Plot (nur valide Daten) -------------------------
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))
            sns.histplot(series, bins=30, ax=axes[0], color="skyblue", kde=True)
            axes[0].set_title(f"Orig: {col}")
            sns.histplot(df_base[new_col].dropna(), bins=30, ax=axes[1], color="lightgreen", kde=True)
            axes[1].set_title(f"Transformed: ln(1 + x)")
            plt.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)
        else:
            base_cols.append(col)

print(f"Log-Report erstellt: {pdf_log_path}")

Log-Report erstellt: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\3_Machine-Learning\3.1_Preprocessing\Preprocessing\2026-02-03_15-26-41\Log_Transformation_Report.pdf


In [8]:
# ------------------------- Gaussian Scaling (QuantileTransformer) -------------------------
# QuantileTransformer kann nicht mit NaN umgehen

df_final = df_base.copy()
final_features = []

pdf_report_path = output_dir / "Gaussian_Scaling_Report.pdf"

with PdfPages(pdf_report_path) as pdf:
    fig = plt.figure(figsize=(10, 6))
    plt.text(0.5, 0.5, "Gaussian Scaling Report\n(Independent Column Processing)", ha='center', va='center', fontsize=16)
    plt.axis('off')
    pdf.savefig(fig)
    plt.close()

    for col in base_cols:
        # ------------------------- valide Daten extrahieren -------------------------
        valid_mask = df_base[col].notna()
        data_valid = df_base.loc[valid_mask, col].values.reshape(-1, 1)
        if len(data_valid) < 10: continue
        
        # ------------------------- Transformieren -------------------------
        scaler = QuantileTransformer(output_distribution='normal', random_state=42)
        trans_vals = scaler.fit_transform(data_valid).flatten()
        
        # ------------------------- Schreiben -------------------------
        new_col_name = f"{col}_gauss"
        df_final.loc[valid_mask, new_col_name] = trans_vals
        final_features.append(new_col_name)
        
        # ------------------------- Plotten -------------------------
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        sns.histplot(data_valid.flatten(), bins=30, ax=axes[0], color="lightgreen", kde=True)
        axes[0].set_title(f"Input: {col}")
        sns.histplot(trans_vals, bins=30, ax=axes[1], color="crimson", kde=True)
        axes[1].set_title(f"Output: Gaussian")
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

print(f"Fertig. Report: {pdf_report_path}")

C:\Users\lucca\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_data.py:2829: UserWarning: n_quantiles (1000) is greater than the total number of samples (373). n_quantiles is set to n_samples.
  warnings.warn(


C:\Users\lucca\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_data.py:2829: UserWarning: n_quantiles (1000) is greater than the total number of samples (397). n_quantiles is set to n_samples.
  warnings.warn(


C:\Users\lucca\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_data.py:2829: UserWarning: n_quantiles (1000) is greater than the total number of samples (257). n_quantiles is set to n_samples.
  warnings.warn(


C:\Users\lucca\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_data.py:2829: UserWarning: n_quantiles (1000) is greater than the total number of samples (12). n_quantiles is set to n_samples.
  warnings.warn(


Fertig. Report: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\3_Machine-Learning\3.1_Preprocessing\Preprocessing\2026-02-03_15-26-41\Gaussian_Scaling_Report.pdf


In [9]:
# ------------------------- Als .csv speichern -------------------------
output_file_som = output_dir / "Preprocessed_SOM_Ready.csv"

meta_cols = ["WGS84_latitude", "WGS84_longitude", "Database_number", "rock_type", "stratigraphic_period"]
meta_cols_present = [c for c in meta_cols if c in df_final.columns]

cols_to_export = list(set(meta_cols_present + final_features + features_to_process))

print(f"Exportiere {len(cols_to_export)} Spalten.")
df_final[cols_to_export].to_csv(output_file_som, index=False)
print(f"Alles gespeichert: {output_file_som}")

Exportiere 51 Spalten.


Alles gespeichert: F:\Abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\3_Machine-Learning\3.1_Preprocessing\Preprocessing\2026-02-03_15-26-41\Preprocessed_SOM_Ready.csv
